# TAE-IA · Module 6 · L24 — «¿Qué suena aquí?» · **the audio half**

| | |
|---|---|
| **Module** | 6 — Practical AI Applications: Images and Audio |
| **Session** | L24 — Track B, closing |
| **What this is** | The skeleton of an app. You write the body. |
| **Due** | Today in class, or as homework with no penalty |
| **Graded** | No. What counts is that it runs and that you show it |

---

## What you are building

An app that **listens to a clip and says what is in it**, using a vocabulary of labels
**you write yourself**.

```
        audio clip
             |
             v
    [ coerce_audio ]  ← the contract: float32 mono at the right rate
             |
      +------+------+
      |             |
      v             v
   [ CLAP ]     [ Whisper ]
    labels      any speech?
      |             |
      +------+------+
             v
      a report in Spanish
             |
             v
        [ Gradio ]
```

On Monday, in L25, **the other half**: the same app takes an image, tags it against
**the same vocabulary**, and compares. Does what you see match what you hear?

> That is why today's contract is not negotiable. Monday's image half plugs into exactly
> the shape today's functions return.

---

## The spec

Your app must, **at minimum**:

1. Accept an audio clip from Gradio and survive anything it is given.
2. Tag it with **CLAP** against a vocabulary you wrote.
3. Use **Whisper** to detect speech, and transcribe it when there is any.
4. Return a report in Spanish with: the top label, its confidence, the top-3, and the
   transcript when it applies.
5. **Refuse to guess** when confidence falls below a floor you set.
6. Run behind a Gradio interface with `gr.Audio` as its input.

And it must **pass the test cell** in section 6. That cell does not get modified.

## The models

| For | Model | Have you used it? |
|---|---|---|
| Tagging sound | `laion/clap-htsat-unfused` | **No. It is new.** Go to HuggingFace and read its card. |
| Detecting speech | Whisper `small` | Yes — L19, L20 |

CLAP does **zero-shot classification**: it has no fixed 50 classes like your L17 model. You hand
it a list of phrases and it says which one fits best. The vocabulary is **yours**, and so are its
mistakes.

## Rules

- **Use whatever you like**: your old notebooks, the documentation, an AI assistant.
- What you cannot do is hand in something that does not run. The test cell decides, not effort.
- If you do not finish today, take what you have and finish it as homework. No penalty.
- **Everyone shows something at the end of L25**, finished or not.

---

## 1 — Install and setup · **given, do not touch**

In [1]:
# Two packages. CLAP rides inside `transformers`, which Colab already has.
!pip install -q openai-whisper librosa

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 32.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import os, sys, gc, time, random
import numpy as np
import torch

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/TAE_IA_M6'
OUTPUT_DIR = f'{DRIVE_ROOT}/L24_output'
INPUT_DIR  = f'{DRIVE_ROOT}/inputs'          # the shared folder from L11/L12
for d in (OUTPUT_DIR, INPUT_DIR):
    os.makedirs(d, exist_ok=True)

# Models live on the runtime disk, not on Drive. Wiped when the runtime recycles
# (~2 min to re-download), and in exchange there is no OSError 95 from symlinks.
MODEL_CACHE = '/content/models'
os.makedirs(MODEL_CACHE, exist_ok=True)
os.environ['HF_HOME']        = MODEL_CACHE
os.environ['TORCH_HOME']     = MODEL_CACHE
os.environ['XDG_CACHE_HOME'] = MODEL_CACHE

if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime > Change runtime type > T4 GPU')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def vram(tag=''):
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  VRAM {used:5.2f} / {total:.1f} GB   {tag}')

print(torch.cuda.get_device_name(0)); vram('empty')

Mounted at /content/drive
Tesla T4
  VRAM  0.00 / 15.6 GB   empty


## 2 — The models · **you load them**

Two models, and together they use under 2 GB of the T4's 15. Nothing to load and free:
both stay resident all day.

**CLAP.** Loads through the `transformers` `pipeline`. The task is called
`zero-shot-audio-classification`. The repo is `laion/clap-htsat-unfused`.

> ⚠️ This repo has **no `.safetensors`** — only `pytorch_model.bin`. If you copy a
> `snapshot_download(..., allow_patterns=['*.safetensors'])` from another notebook, nothing
> downloads and the error surfaces much later.

**Whisper.** Same as L19: `whisper.load_model('small', download_root=MODEL_CACHE)`.

In [3]:
from transformers import pipeline
import whisper

# CLAP performs zero-shot audio classification with our own labels.
tagger = pipeline(
    'zero-shot-audio-classification',
    model='laion/clap-htsat-unfused',
    device=0
)

vram('after CLAP')        # expect ~0.6 GB

# Whisper transcribes speech in Spanish or English.
asr = whisper.load_model('small', download_root=MODEL_CACHE)

vram('both loaded')       # expect ~1.6 GB

config.json:   0%|          | 0.00/5.39k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  615MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/447 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

  VRAM  0.62 / 15.6 GB   after CLAP




  0%|                                               | 0.00/461M [00:00<?, ?iB/s]

  2%|▉                                      | 11.0M/461M [00:00<00:04, 115MiB/s]

  6%|██▌                                    | 29.6M/461M [00:00<00:02, 162MiB/s]

 10%|███▉                                   | 46.2M/461M [00:00<00:02, 168MiB/s]

 13%|█████▎                                 | 62.2M/461M [00:00<00:02, 166MiB/s]

 17%|██████▊                                | 80.4M/461M [00:00<00:02, 175MiB/s]

 22%|████████▊                               | 101M/461M [00:00<00:01, 190MiB/s]

 26%|██████████▌                             | 122M/461M [00:00<00:01, 197MiB/s]

 30%|███████████▉                           | 140M/461M [00:02<00:12, 26.5MiB/s]

 33%|████████████▉                          | 154M/461M [00:02<00:10, 32.0MiB/s]

 37%|██████████████▎                        | 169M/461M [00:03<00:07, 41.2MiB/s]

 40%|███████████████▌                       | 183M/461M [00:03<00:05, 52.1MiB/s]

 43%|█████████

  VRAM  1.59 / 15.6 GB   both loaded


## 3 — The contract · **given, and not modified**

This cell is half of the contract Monday's session closes. It is the only code you are handed,
and it is handed to you for one reason: **this is where the failure that does not crash lives.**

Three measured facts about CLAP, not opinions:

| Give it… | What happens |
|---|---|
| `int16` exactly as Gradio delivers it | Classifies rain as **"a dog barking", 0.576**. No exception. |
| The wrong sample rate | **Sometimes** right. Dog breaks at 22 kHz and 16 kHz; rain, birds, vacuum and clock all survive. |
| A `dict` with `sampling_rate` | `TypeError`. CLAP **cannot** know your rate — resampling is your job. |

The second is the dangerous one: five clips out of six forgive the bug, so your own test passes
and the person next to you gets nonsense.

In [4]:
import librosa

# The rate each stage wants. Two different numbers in one pipeline.
CLAP_SR     = 48000     # measured: at any other rate CLAP fails intermittently
ASR_SR      = 16000     # Whisper
MAX_SECONDS = 60        # a 30-minute upload is the #1 cause of a stalled demo
MIN_SECONDS = 0.5


def list_inputs():
    """What is in the shared folder right now."""
    names = sorted(f for f in os.listdir(INPUT_DIR) if not f.startswith('.'))
    print(f'{INPUT_DIR}  ({len(names)} files)')
    for n in names:
        print(f'  {os.path.getsize(os.path.join(INPUT_DIR, n))/1e6:6.2f} MB  {n}')
    return names


def upload_inputs():
    """Pick files from your machine; they land in Drive and stay there."""
    from google.colab import files
    for fname, data in files.upload().items():
        with open(os.path.join(INPUT_DIR, fname), 'wb') as f:
            f.write(data)
        print(f'  saved {fname}')


def load_audio(name_or_path, sr=None):
    """A name in the inputs folder, or any path -> (wav float32 mono, sr)."""
    path = name_or_path
    if not os.path.isabs(path):
        path = os.path.join(INPUT_DIR, name_or_path)
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'{name_or_path} is not in {INPUT_DIR}. '
            'Run upload_inputs() and pick it, or copy it into that folder on Drive.')
    wav, got_sr = librosa.load(path, sr=sr, mono=True)
    return wav.astype('float32'), got_sr


def coerce_audio(wav, sr, target_sr):
    """Anything a user can hand us -> float32 mono at target_sr, or ValueError.

    This function is the contract. Call it ALWAYS, before a model sees anything.
    """
    if wav is None:
        raise ValueError('No audio received. Please upload a clip.')
    wav = np.asarray(wav)
    if wav.dtype.kind in 'iu':                       # gradio hands back int16
        wav = wav.astype('float32') / np.iinfo(wav.dtype).max
    wav = wav.astype('float32')
    if wav.ndim > 1:                                 # to mono, either layout
        wav = wav.mean(axis=0) if wav.shape[0] < wav.shape[1] else wav.mean(axis=1)
    if wav.size < sr * MIN_SECONDS:
        raise ValueError(f'Clip too short ({wav.size/sr:.2f} s). Use at least {MIN_SECONDS} s.')
    if wav.size > sr * MAX_SECONDS:
        wav = wav[:int(sr * MAX_SECONDS)]            # truncate, and the caller says so
    if np.abs(wav).max() < 1e-4:
        raise ValueError('That clip is silent. Whisper would invent words for it.')
    if sr != target_sr:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr)
    return wav.astype('float32'), target_sr


print('contract loaded. CLAP_SR =', CLAP_SR, '| ASR_SR =', ASR_SR)

contract loaded. CLAP_SR = 48000 | ASR_SR = 16000


### The shape your functions return · **given, and it is the joint with L25**

On Monday you write `analyse_image(...)`. It returns **a dict with these same keys**, and L25's
join is comparing the two `tags` lists. Return a different shape today and Monday does not fit.

```python
{
    'modality': 'audio',            # Monday: 'image'
    'scene'   : 'casa',             # which vocabulary was used
    'tags'    : [('lluvia', 0.91), ('viento', 0.05), ...],   # sorted, highest first
    'text'    : 'what the speech said, or None',
    'summary' : 'one sentence in Spanish',
}
```

## 4 — Your vocabulary · **you write it**

Here is the difference from the L17 classifier. That one had 50 classes somebody else trained.
This one has **whatever you write**, and it changes its answer when you change the wording.

### How that actually works

Your L17 model ends in fifty output neurons. Those fifty were decided before training, and a
fifty-first means training again.

CLAP has **no output neurons at all**. It turns the audio into a vector, turns each of your
phrases into a vector *in the same space*, and answers with whichever is closest.

```
  audio  ──►  ●                     your phrases ──►  ● ● ●
                 nearest one wins
```

So "the classes" are a Python list of strings you pass at call time. Nothing is trained, nothing
is saved, and you can change the whole vocabulary between two calls. This is the same trick CLIP
does for images — which is exactly why Monday's half is short.

Cost does not depend on the list: a few labels or a few dozen, a one-second clip or a
thirty-second one, inference is ~0.15 s either way.

Two measured things worth knowing:

- `'a dog barking'` scores 1.000 where `'dog'` scores 0.983. Descriptive phrases beat bare
  words — CLAP was trained on descriptions, not on labels.
- **CLAP never says "none of these".** A dog, offered only rain / vacuum / birds / engine,
  answers **"birds chirping" at 0.497**. It always returns a ranked list. That is why the spec
  asks for a confidence floor: the "I don't know" is yours to write.

Write **at least two scenes** with **at least four labels each**. Pick sounds you can actually
record or find today — you need them to test.

> **No clips of your own?** There are nine in the shared `inputs/` folder: dog, rain, birds,
> vacuum cleaner, crying baby, ticking clock, a chainsaw, and speech in Spanish and English.
> Run `list_inputs()` to see them. Using your own recordings is better, but not at the cost of
> having nothing to test against.

In [5]:
# The phrases are in English because CLAP performs better with English descriptions.
SCENES = {
    'general': [
        'a dog barking',
        'rain falling',
        'birds chirping',
        'a vacuum cleaner running',
        'a baby crying',
        'a clock ticking',
        'a chainsaw running',
        'a person speaking in Spanish',
        'a person speaking in English',
        'something else or an unknown sound',
    ],
    'aula': [
        'students talking in a classroom',
        'a teacher speaking',
        'a school bell ringing',
        'chairs moving on the floor',
        'pages turning in a book',
        'students applauding',
        'something else or an unknown sound',
    ],
    'musica': [
        'a mariachi band playing with trumpets and violins',
        'heavy metal music with distorted guitars and fast drums',
        'acoustic guitar music',
        'classical piano music',
        'electronic dance music',
        'something else or unknown music',
    ],
}

# Below this score, the app refuses to name the sound.
# Do not copy this number from anyone: measure it. Run your clips, see what a hit scores
# and what a miss scores, and put the floor between them.
#
# And know what the floor does NOT do. Measured: a chainsaw, offered a list with no chainsaw
# in it, comes back "a vacuum cleaner" at 0.880 — above any floor you would reasonably set.
# The score is a softmax over YOUR list, so it says "which of these fits best", never "does
# any of these fit". A floor catches the model being unsure. It cannot catch it being wrong.
# If that bothers you, the fixes are a wider vocabulary, an explicit "something else" label,
# or looking at the gap between the top two instead of the top one.
CONF_FLOOR = 0.40  # Initial value; adjust it after measuring real clips.

## 5 — The four functions · **you write them**

The signatures and docstrings are given so the joint with L25 works. The bodies are not.

Suggested order: `tag_audio` first, and test it alone on a real clip before going further.
If that one does not give sensible answers, none of the other three can.

In [6]:
def tag_audio(wav, sr, labels):
    """(wav, sr) + a list of phrases -> [(label, score), ...] sorted, highest first.

    Assumes sr == CLAP_SR. If it is not, that is a bug in the caller: use coerce_audio.
    CLAP returns a list of dicts with 'label' and 'score'.
    """
    assert sr == CLAP_SR, f'{sr} != {CLAP_SR} - coerce first'
    results = tagger(wav, candidate_labels=labels)
    return [(item['label'], float(item['score'])) for item in results]


def transcribe(wav, sr, language=None):
    """(wav, sr) -> text. Whisper wants float32 at 16 kHz."""
    assert sr == ASR_SR, f'{sr} != {ASR_SR} - coerce first'
    result = asr.transcribe(wav, language=language, fp16=torch.cuda.is_available())
    return result['text'].strip()


def analyse_audio(wav, sr, scene):
    """(wav, sr) + a scene name -> the dict from section 3b.

    Your logic goes here: tag it, decide whether there is speech, apply the confidence floor,
    and compose `summary` in Spanish. You decide what counts as "there is speech".
    """
    if scene not in SCENES:
        raise ValueError(f'Unknown scene: {scene}. Choose one of {list(SCENES)}.')

    labels = SCENES[scene]
    w48, _ = coerce_audio(wav, sr, CLAP_SR)
    tags = tag_audio(w48, CLAP_SR, labels)

    top_label, top_score = tags[0]
    speech_words = ('speaking', 'talking', 'speech', 'voice')
    speech_score = max(
        (score for label, score in tags
         if any(word in label.lower() for word in speech_words)),
        default=0.0
    )

    text = None
    if speech_score >= CONF_FLOOR:
        w16, _ = coerce_audio(wav, sr, ASR_SR)
        candidate_text = transcribe(w16, ASR_SR)
        if candidate_text:
            text = candidate_text

    if top_score < CONF_FLOOR or 'unknown' in top_label.lower():
        summary = 'No pude identificar este sonido con suficiente confianza.'
    elif text:
        summary = f'Escuché voz y la transcripción es: {text}'
    else:
        summary = f'El sonido principal parece ser: {top_label}.'

    return {
        'modality': 'audio',
        'scene': scene,
        'tags': tags,
        'text': text,
        'summary': summary,
    }


def run_audio(x, sr=None, scene='casa'):
    """The one function the UI calls. Anything in, and it NEVER raises.

    Returns (summary, report), where `report` is text for the screen — labels, timings,
    whatever helps you debug live. If something fails, return a message a user can
    understand, not a traceback.
    """
    started = time.time()
    try:
        if isinstance(x, str):
            wav, sr = load_audio(x, sr=None)
        else:
            wav = x
            if sr is None:
                raise ValueError('No sample rate was received.')

        wav, sr = coerce_audio(wav, sr, sr)
        result = analyse_audio(wav, sr, scene)

        top3 = result['tags'][:3]
        tag_lines = '\n'.join(
            f'  {i}. {label}: {score:.3f}'
            for i, (label, score) in enumerate(top3, start=1)
        )
        transcript = result['text'] or 'No speech detected'
        report = (
            f"Scene: {result['scene']}\n"
            f"Summary: {result['summary']}\n"
            f"Top 3:\n{tag_lines}\n"
            f"Transcript: {transcript}\n"
            f"Time: {time.time() - started:.2f} s"
        )
        return result['summary'], report

    except ValueError as error:
        message = str(error)
        return message, message
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        message = 'The GPU ran out of memory. Try a shorter clip.'
        return message, message
    except Exception as error:
        print(f'[run_audio] {type(error).__name__}: {error}')
        message = 'Something went wrong. Try another audio clip.'
        return message, message

## 6 — The test cell · **given. It has to pass.**

Do not modify it. If it fails, the problem is in your code, not in the test.

It checks three things: that hostile inputs do not raise, that the contract actually protects
you from the `int16` bug, and that your labels fire on your clips.

In [7]:
# ---------- A. hostile inputs: none of these may raise ----------
rng   = np.random.default_rng(SEED)
noise = (0.2 * rng.standard_normal(CLAP_SR * 5)).astype('float32')
scene = next(iter(SCENES))

print('--- hostile inputs ---')
cases = [
    ('None',        None,                                          CLAP_SR),
    ('silence',     np.zeros(CLAP_SR * 5, dtype='float32'),        CLAP_SR),
    ('stereo',      np.stack([noise, noise], axis=-1),             CLAP_SR),
    ('int16',       (noise * 32767).astype('int16'),               CLAP_SR),
    ('odd rate',    noise,                                         44100),
    ('0.1 s',       noise[:int(CLAP_SR * 0.1)],                    CLAP_SR),
    ('30 minutes',  np.tile(noise, 360),                           CLAP_SR),
]
fails = 0
for tag, bad, sr in cases:
    try:
        _s, msg = run_audio(bad, sr=sr, scene=scene)
        first = (msg or _s or '').splitlines()[0][:60]
        print(f'  {tag:12s} -> {first}')
    except Exception as e:
        print(f'  {tag:12s} -> RAISED {type(e).__name__}: {e}')
        fails += 1

# ---------- B. the contract is worth something ----------
print('\n--- int16, with and without the contract ---')
i16 = (noise * 32767).astype('int16')
raw  = tag_audio(i16.astype('float32'), CLAP_SR, SCENES[scene])      # unnormalised: wrong
good = tag_audio(*coerce_audio(i16, CLAP_SR, CLAP_SR), SCENES[scene])
print(f'  without coerce_audio : {raw[0][0]:24s} {raw[0][1]:.3f}')
print(f'  with coerce_audio    : {good[0][0]:24s} {good[0][1]:.3f}')
print('  (on white noise the two may agree; on a real clip they almost never do)')

# ---------- C. your clips against your vocabulary ----------
print('\n--- your clips ---')
clips = [f for f in list_inputs() if f.lower().endswith(('.wav', '.mp3', '.ogg', '.flac'))]
if not clips:
    print('  (no clips in the folder - run upload_inputs())')
for name in clips[:4]:
    wav, sr = load_audio(name)
    summary, report = run_audio(wav, sr=sr, scene=scene)
    print(f'  {name}: {summary}')

vram('peak')
print(f"\npeak {torch.cuda.max_memory_allocated()/1e9:.2f} GB of "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print('\nhostile inputs that raised:', fails, '(must be 0)')

--- hostile inputs ---
  None         -> No audio received. Please upload a clip.
  silence      -> That clip is silent. Whisper would invent words for it.
  stereo       -> Scene: general
  int16        -> Scene: general
  odd rate     -> Scene: general
  0.1 s        -> Clip too short (0.10 s). Use at least 0.5 s.
  30 minutes   -> Scene: general

--- int16, with and without the contract ---
  without coerce_audio : a dog barking            0.401
  with coerce_audio    : rain falling             0.573
  (on white noise the two may agree; on a real clip they almost never do)

--- your clips ---
/content/drive/MyDrive/TAE_IA_M6/inputs  (19 files)
    0.03 MB  01_person.jpg
    0.04 MB  02_car.jpg
    0.02 MB  03_dog.jpg
    0.03 MB  04_busy_street.jpg
    0.01 MB  05_violin.jpg
    0.01 MB  06_bird.jpg
    0.03 MB  07_fishing.jpg
    0.00 MB  README.md
    0.00 MB  _manifest.tsv
    0.44 MB  birds.wav
    0.44 MB  chainsaw.wav
    0.44 MB  clock_tick.wav
    0.44 MB  crying_baby.wav
  

## 7 — The app · **you write it**

Everything you need from Gradio you already used in L11 and L12: `Blocks`, `Row`, `Column`,
`Button`, `Textbox`, `Dropdown`, `Markdown`, `.click()`, `.queue()`, `.launch()`.

**The only new thing is `gr.Audio`**, and it carries the trap you saw in the demo:

```python
gr.Audio(type='numpy')   # the callback receives (sr, wav). Rate FIRST.
```

`librosa.load` returns `(wav, sr)`. Gradio returns `(sr, wav)`. Unpack it backwards and nothing
raises — you resample to the length of the array and listen to noise.

Minimum interface requirements:

- `gr.Audio(type='numpy')` as input
- a `gr.Dropdown` to pick the scene, fed from `SCENES`
- a `gr.Textbox` with the report
- a button, and `.queue()` before `.launch(share=True)`

In [8]:
import gradio as gr
print('gradio', gr.__version__)

def on_click(audio, scene):
    if audio is None:
        return run_audio(None, sr=CLAP_SR, scene=scene)
    sr, wav = audio                 # Gradio returns rate FIRST
    return run_audio(wav, sr=sr, scene=scene)

with gr.Blocks(title='¿Qué suena aquí?') as demo:
    gr.Markdown('# ¿Qué suena aquí?')
    gr.Markdown('Upload or record a clip. The app tags the sound and detects speech.')

    with gr.Row():
        with gr.Column():
            audio_input = gr.Audio(type='numpy', label='Audio clip')
            scene_input = gr.Dropdown(
                choices=list(SCENES.keys()),
                value=next(iter(SCENES)),
                label='Vocabulary scene'
            )
            analyse_button = gr.Button('Analyse audio', variant='primary')

        with gr.Column():
            summary_output = gr.Textbox(label='Spanish summary', lines=3)
            report_output = gr.Textbox(label='Detailed report', lines=10)

    analyse_button.click(
        fn=on_click,
        inputs=[audio_input, scene_input],
        outputs=[summary_output, report_output]
    )

demo.queue().launch(share=True, quiet=True)

gradio 6.26.0
* Running on public URL: https://67eafb7f76abf24ed8.gradio.live


In [9]:
# Always close before relaunching, or you leak ports.
# demo.close()

---

## If you get stuck

**The fastest way to understand the vocabulary** is to stop reading and run one clip twice,
against two different lists:

```python
wav, sr = coerce_audio(*load_audio('dog.wav'), CLAP_SR)
print(tag_audio(wav, sr, ['a dog barking', 'rain falling']))
print(tag_audio(wav, sr, ['a wolf howling', 'a person laughing']))
```

Same audio, different answers, nothing retrained. Thirty seconds, and the idea lands harder
than any paragraph above.

**If an assistant is writing your CLAP code**, check it against section 2 before you run it.
CLAP is recent and less common than Whisper, and assistants confidently invent APIs for it —
a `CLAPModel.classify()` that does not exist, a `sampling_rate=` argument the pipeline rejects.
The docstrings in this notebook are the ground truth; a fluent answer is not.

**Work out which half is broken** before changing anything. `tag_audio` on a known clip tells
you whether the model or your composition is at fault, and those need different fixes.

---

## Before you leave

- [X] The test cell runs and reports **0** hostile inputs that raised
- [X] `analyse_audio` returns the dict with all five keys from section 3b
- [X] The app opens, accepts a clip, and answers something sensible
- [X] `CONF_FLOOR` holds a number you **measured**, not one you copied
- [X] The notebook is saved to Drive
- [X] **Restart the runtime and run everything again.** A notebook that only runs in the
      current state of memory is not a deliverable, it is a memory.

## Monday (L25)

You get the other half of the skeleton: `analyse_image`, against **the same `SCENES`**. The join
is comparing the `tags` from both halves and saying whether they agree. Then the showcase.

If you did not finish today, Monday starts with finishing. Whatever does not fit goes home as
homework, with no penalty — but everyone shows something.

---
*TAE-IA 2025 · Cocyten-Nayarit · Module 6 · L24*